# 03 — Modeling (Model training)

Trains the classical ML models from the full pipeline with **10-fold Stratified Cross-Validation**:

- Logistic Regression — text-only features *and* text + URL-domain features
- Random Forest
- Naive Bayes
- SVM (linear kernel)

Features: TF-IDF (unigrams+bigrams, 20k max features) optionally combined with domain features via `MultiLabelBinarizer` + `hstack`. Deep-learning models (CNN/LSTM) and BERTweet are covered in `04_evaluation.ipynb`.

## Imports & setup

In [ ]:
!pip install nltk
!pip show nltk
!pip install tldextract
!pip install datasets
!pip install shap

In [ ]:
import os
import requests
import json
import time
import zipfile

import nltk
import kagglehub
import re

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns
import pandas as pd
import numpy as np

from collections import Counter

import tldextract

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
from sklearn.svm import SVC

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.mixed_precision import set_global_policy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense, SpatialDropout1D, Dropout, BatchNormalization, Bidirectional, Attention
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from keras.metrics import AUC, Precision, Recall

import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from tensorflow.keras.optimizers import Adam
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

from imblearn.over_sampling import SMOTE
import shap

In [ ]:
import sys, os

PROJECT_PATH = "/content/Sexism-Classification" if os.path.exists("/content/Sexism-Classification") else os.getcwd()
sys.path.append(PROJECT_PATH)

## Getting the data

In [ ]:
# Download latest version
path = kagglehub.dataset_download("aadyasingh55/sexism-detection-in-english-texts")


print("Path to dataset files:", path)

dev_df = pd.read_csv(f"{path}/dev.csv")
test_df = pd.read_csv(f"{path}/test (1).csv")
train_df = pd.read_csv(f"{path}/train (2).csv")


## Preprocessing

(Uses the same domain-aware cleaning developed in `02_preprocessing.ipynb`.)

In [ ]:
# Initialise NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Function to clean text and extract domains
def clean_text(text, lowercase=True, replace_urls=True, extract_domain=False, remove_stopwords=True, lemmatize=True):
    if lowercase:
        text = text.lower()

    # Extract domain names from URLs
    url_pattern = r'https?://\S+|www\.\S+'
    domains = []  # Store extracted domains
    matches = re.findall(url_pattern, text)

    for match in matches:
        extracted = tldextract.extract(match)
        domain = f"{extracted.domain}.{extracted.suffix}"  # e.g., "cnn.com"
        domains.append(domain)  # Save domain for analysis
        text = text.replace(match, "")  # Remove the URL from text

    # Remove special characters, punctuation, and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    if lemmatize:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens), domains  # Convert tokens back to string


In [ ]:
# Apply function to dataset
train_dev_data[["text", "domains"]] = pd.DataFrame(train_dev_data["text"].apply(clean_text).tolist(), index=train_dev_data.index)

# Process the separate test set as well
test_df[["text", "domains"]] = pd.DataFrame(test_df["text"].apply(clean_text).tolist(), index=test_df.index)

print(train_dev_data.head())


## Prep

# Ensures mixed precision is set globally (allows for 16bit and 32bit float types (runs faster uses less memory))
set_global_policy('mixed_float16')

##Prep

### Intitalisation

### Initialisation

In [ ]:
# Initialize MultiLabelBinarizer for domains (fit on combined train_dev_data)
mlb = MultiLabelBinarizer()
domain_features_train = mlb.fit_transform(train_dev_data["domains"])

# Vectorize the cleaned text using TF-IDF (fit on combined train_dev_data)
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_text_train = vectorizer.fit_transform(train_dev_data["text"])
X_combined_train = hstack([X_text_train, domain_features_train])

# Encode the target variable for deep learning models
label_encoder = LabelEncoder()
y_encoded_train = label_encoder.fit_transform(train_dev_data['label_sexist'])


# --- Prepare the separate TEST set features ---
domain_features_test = mlb.transform(test_df["domains"]) # Use fitted MLB
X_text_test = vectorizer.transform(test_df["text"]) # Use fitted TF-IDF Vectorizer
X_combined_test = hstack([X_text_test, domain_features_test])
y_encoded_test = label_encoder.transform(test_df['label_sexist'])


# Tokenize for deep learning models (fit on combined train_dev_data)
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_dev_data['text'])
X_seq_train = tokenizer.texts_to_sequences(train_dev_data['text'])

VOCAB_SIZE_DEEP_LEARNING = len(tokenizer.word_index) + 1
if tokenizer.num_words is not None:
    VOCAB_SIZE_DEEP_LEARNING = min(VOCAB_SIZE_DEEP_LEARNING, tokenizer.num_words + 1)

# Determine maxlen for padding based on train_dev_data
percentile_95 = int(np.percentile([len(seq) for seq in X_seq_train], 95))
MAX_SEQUENCE_LENGTH = percentile_95
X_pad_train = pad_sequences(X_seq_train, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# Tokenize and pad the separate TEST set for deep learning models
X_seq_test = tokenizer.texts_to_sequences(test_df['text'])
X_pad_test = pad_sequences(X_seq_test, maxlen=MAX_SEQUENCE_LENGTH, padding='post')




### Storage

### Storage

In [ ]:
# Define common variables for cross-validation
n_splits = 10 # Number of folds for StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42) # Set random_state for reproducibility of splits

# Initialise results
results = {
  "LR_Text": [],
  "LR_Domains": [],
  "NB": [],
  "RF": [],
  "SVM": [],
  "LSTM": [],
  "CNN": [],
  "BERT": [],
  "NBft": [],
}

# Initialize history storage for deep learning models (for plotting average curves)
avg_histories = {
    "CNN": {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
        'precision': [], 'recall': [], 'val_precision': [], 'val_recall': [],
        'learning_rate': []},
    "LSTM": {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
        'precision': [], 'recall': [], 'val_precision': [], 'val_recall': [],
        'learning_rate': []},
    "BERT": {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "learning_rate": []},
}

# Initialize lists for additional metrics for each model
precision_list_lr_text, recall_list_lr_text, f1_list_lr_text = [], [], []
precision_list_lr_combined, recall_list_lr_combined, f1_list_lr_combined = [], [], []
precision_list_nb, recall_list_nb, f1_list_nb = [], [], []
precision_list_rf, recall_list_rf, f1_list_rf = [], [], []
precision_list_svm, recall_list_svm, f1_list_svm = [], [], []
precision_list_cnn, recall_list_cnn, f1_list_cnn = [], [], []
precision_list_lstm, recall_list_lstm, f1_list_lstm = [], [], []
precision_list_bert, recall_list_bert, f1_list_bert = [], [], []
precision_list_nbft, recall_list_nbft, f1_list_nbft = [], [], []

# Initialize confusion matrix sums (ensure they are reset or properly scoped for each model)
conf_matrix_sum_lr_text = np.zeros((2, 2))
conf_matrix_sum_lr_combined = np.zeros((2, 2))
conf_matrix_sum_nb = np.zeros((2, 2))
conf_matrix_sum_rf = np.zeros((2, 2))
conf_matrix_sum_svm = np.zeros((2, 2))
conf_matrix_sum_cnn = np.zeros((2, 2))
conf_matrix_sum_lstm = np.zeros((2, 2))
conf_matrix_sum_bert = np.zeros((2, 2))
conf_matrix_sum_nbft = np.zeros((2, 2))


## Machine Learning models

## Machine Learning models

###Logisitc Regression

In [ ]:
print("\n--- Training Logistic Regression Models ---")

# Separate loop for Logistic Regression
for fold, (train_index, test_index) in enumerate(skf.split(X_text_train, y_encoded_train)):
    print(f"\nLogistic Regression - Fold {fold + 1}/{n_splits}...")

    # Ensure X_text_train_dev and X_combined_train_dev are in a sliceable sparse format (like CSR)
    X_text_train_csr = X_text_train.tocsr()
    X_combined_train_csr = X_combined_train.tocsr()


    # Splits for text-only Logistic Regression
    X_train_lr_text, X_test_lr_text = X_text_train_csr[train_index], X_text_train_csr[test_index]
    y_train_lr, y_test_lr = y_encoded_train[train_index], y_encoded_train[test_index]

    # Splits for combined-features Logistic Regression
    X_train_lr_combined, X_test_lr_combined = X_combined_train_csr[train_index], X_combined_train_csr[test_index]
    # y_train_lr and y_test_lr are the same for combined model

    # Logistic Regression (Text only)
    lr_model_t = LogisticRegression(max_iter=1000, class_weight='balanced')
    lr_model_t.fit(X_train_lr_text, y_train_lr)
    y_pred_t = lr_model_t.predict(X_test_lr_text)
    acc_t = accuracy_score(y_test_lr, y_pred_t)
    results["LR_Text"].append(acc_t)

    precision_t, recall_t, f1_t, _ = precision_recall_fscore_support(y_test_lr, y_pred_t, average='binary', pos_label=1)
    precision_list_lr_text.append(precision_t)
    recall_list_lr_text.append(recall_t)
    f1_list_lr_text.append(f1_t)
    conf_matrix_sum_lr_text += confusion_matrix(y_test_lr, y_pred_t)

    print(f"  LR (Text Only) - Accuracy: {acc_t:.4f}, Precision: {precision_t:.4f}, Recall: {recall_t:.4f}, F1 Score: {f1_t:.4f}")

    # Logistic Regression (Combined features)
    lr_model_c = LogisticRegression(max_iter=1000, class_weight='balanced', )
    lr_model_c.fit(X_train_lr_combined, y_train_lr)
    y_pred_c = lr_model_c.predict(X_test_lr_combined)
    acc_c = accuracy_score(y_test_lr, y_pred_c)
    results["LR_Domains"].append(acc_c)

    precision_c, recall_c, f1_c, _ = precision_recall_fscore_support(y_test_lr, y_pred_c, average='binary', pos_label=1)
    precision_list_lr_combined.append(precision_c)
    recall_list_lr_combined.append(recall_c)
    f1_list_lr_combined.append(f1_c)
    conf_matrix_sum_lr_combined += confusion_matrix(y_test_lr, y_pred_c)

    print(f"  LR (Domains) - Accuracy: {acc_c:.4f}, Precision: {precision_c:.4f}, Recall: {recall_c:.4f}, F1 Score: {f1_c:.4f}")


# Calculate and print average results for LR (including new metrics)
avg_accuracy_T = np.mean(results["LR_Text"])
std_accuracy_T = np.std(results["LR_Text"])
avg_precision_T = np.mean(precision_list_lr_text)
avg_recall_T = np.mean(recall_list_lr_text)
avg_f1_T = np.mean(f1_list_lr_text)

avg_accuracy_C = np.mean(results["LR_Domains"])
std_accuracy_C = np.std(results["LR_Domains"])
avg_precision_C = np.mean(precision_list_lr_combined)
avg_recall_C = np.mean(recall_list_lr_combined)
avg_f1_C = np.mean(f1_list_lr_combined)


# Text Only
print(f"\nAverage Accuracy for Logistic Regression (Text only) over {n_splits} folds: {avg_accuracy_T:.4f} ± {std_accuracy_T:.4f}")
print(f"  Avg Precision: {avg_precision_T:.4f}, Avg Recall: {avg_recall_T:.4f}, Avg F1 Score: {avg_f1_T:.4f}")

# Plotting average confusion matrix for LR_Combined
avg_conf_matrix_lr_text = conf_matrix_sum_lr_text / n_splits
conf_matrix_custom_lr_text = np.array([[avg_conf_matrix_lr_text[1, 1], avg_conf_matrix_lr_text[0, 1]],
                                          [avg_conf_matrix_lr_text[1, 0], avg_conf_matrix_lr_text[0, 0]]], dtype=int)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_lr_text, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"],
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Logistic Regression (Text) Average Confusion Matrix")
plt.show()
print("Logistic Regression (Text) Average Confusion Matrix:\n", conf_matrix_custom_lr_text)


# Domains included
print(f"\nAverage Accuracy for Logistic Regression (with Domains) over {n_splits} folds: {avg_accuracy_C:.4f} ± {std_accuracy_C:.4f}")
print(f"  Avg Precision: {avg_precision_C:.4f}, Avg Recall: {avg_recall_C:.4f}, Avg F1 Score: {avg_f1_C:.4f}")

# Plotting average confusion matrix for LR_Combined
avg_conf_matrix_lr_combined = conf_matrix_sum_lr_combined / n_splits
conf_matrix_custom_lr_combined = np.array([[avg_conf_matrix_lr_combined[1, 1], avg_conf_matrix_lr_combined[0, 1]],
                                          [avg_conf_matrix_lr_combined[1, 0], avg_conf_matrix_lr_combined[0, 0]]], dtype=int)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_lr_combined, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"],
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Logistic Regression (Domains) Average Confusion Matrix")
plt.show()
print("Logistic Regression (Domains) Average Confusion Matrix:\n", conf_matrix_custom_lr_combined)

### Random Forrest

In [ ]:
print("\n--- Training Random Forest Model ---")

# Separate loop for Random Forest
for fold, (train_index, test_index) in enumerate(skf.split(X_text_train, train_dev_data["label_sexist"])):
    print(f"\n Random Forest - Fold {fold + 1}/{n_splits}...")

    # Ensure X_text_train_dev is in CSR format
    X_text_train_csr = X_text_train.tocsr()

    # Splits for Random Forest (uses text only)
    X_train_rf, X_test_rf = X_text_train_csr[train_index], X_text_train_csr[test_index]
    y_train_rf, y_test_rf = y_encoded_train[train_index], y_encoded_train[test_index]

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train_rf, y_train_rf)
    y_pred_rf = rf_model.predict(X_test_rf)
    acc_rf = accuracy_score(y_test_rf, y_pred_rf)

    # Calculate precision, recall, f1 (ensure pos_label matches encoded value for 'sexist')
    # Assuming 'sexist' is encoded as 1 and 'not sexist' as 0
    precision, recall, f1, _ = precision_recall_fscore_support(y_test_rf, y_pred_rf, average='binary', pos_label=1)
    precision_list_rf.append(precision)
    recall_list_rf.append(recall)
    f1_list_rf.append(f1)

    results["RF"].append(acc_rf)
    print(f"  Accuracy: {acc_rf:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")

    # Accumulate confusion matrix
    conf_matrix_sum_rf += confusion_matrix(y_test_rf, y_pred_rf)



# Calculate and print average results for RF
avg_accuracy_rf = np.mean(results["RF"])
std_accuracy_rf = np.std(results["RF"])
print(f"\nAverage Accuracy for Random Forest (RF) over {n_splits} folds: {avg_accuracy_rf:.4f} ± {std_accuracy_rf:.4f}")

# Plotting average confusion matrix for RF (same as your original, just use the _rf suffix)
avg_conf_matrix_rf = conf_matrix_sum_rf / n_splits
conf_matrix_custom_rf = np.array([[avg_conf_matrix_rf[1, 1], avg_conf_matrix_rf[0, 1]],
                                 [avg_conf_matrix_rf[1, 0], avg_conf_matrix_rf[0, 0]]], dtype=int)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_rf, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"], # Corrected labels
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"]) # Corrected labels
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Random Forest Average Confusion Matrix")
plt.show()

print("Random Forest Average Confusion Matrix:\n", conf_matrix_custom_rf)

### Naive Bayes

In [ ]:
print("\n--- Training Naive Bayes Model ---")

# Separate loop for Naive Bayes
for fold, (train_index, test_index) in enumerate(skf.split(X_text_train, y_encoded_train)):
    print(f"\nNaive Bayes - Fold {fold + 1}/{n_splits}...")

    # Ensure X_text_train_dev is in CSR format
    X_text_train_csr = X_text_train.tocsr()

    # Splits for Naive Bayes (uses text only)
    X_train_nb, X_test_nb = X_text_train_csr[train_index], X_text_train_csr[test_index]
    y_train_nb, y_test_nb = y_encoded_train[train_index], y_encoded_train[test_index]

    nb_model = MultinomialNB()
    nb_model.fit(X_train_nb, y_train_nb)
    y_pred_nb = nb_model.predict(X_test_nb)
    acc_nb = accuracy_score(y_test_nb, y_pred_nb)
    results["NB"].append(acc_nb)

    precision_nb_f, recall_nb_f, f1_nb_f, _ = precision_recall_fscore_support(y_test_nb, y_pred_nb, average='binary', pos_label=1)
    precision_list_nb.append(precision_nb_f)
    recall_list_nb.append(recall_nb_f)
    f1_list_nb.append(f1_nb_f)
    conf_matrix_sum_nb += confusion_matrix(y_test_nb, y_pred_nb)

    print(f"  Naive Bayes - Accuracy: {acc_nb:.4f}, Precision: {precision_nb_f:.4f}, Recall: {recall_nb_f:.4f}, F1 Score: {f1_nb_f:.4f}")

# Calculate and print average results for NB
avg_accuracy_nb = np.mean(results["NB"])
std_accuracy_nb = np.std(results["NB"])
avg_precision_nb = np.mean(precision_list_nb)
avg_recall_nb = np.mean(recall_list_nb)
avg_f1_nb = np.mean(f1_list_nb)

print(f"\nAverage Accuracy for Naive Bayes (NB) over {n_splits} folds: {avg_accuracy_nb:.4f} ± {std_accuracy_nb:.4f}")
print(f"  Avg Precision: {avg_precision_nb:.4f}, Avg Recall: {avg_recall_nb:.4f}, Avg F1 Score: {avg_f1_nb:.4f}")

# Plotting average confusion matrix for NB
avg_conf_matrix_nb = conf_matrix_sum_nb / n_splits
conf_matrix_custom_nb = np.array([[avg_conf_matrix_nb[1, 1], avg_conf_matrix_nb[0, 1]],
                                 [avg_conf_matrix_nb[1, 0], avg_conf_matrix_nb[0, 0]]], dtype=int)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_nb, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"],
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Naive Bayes Average Confusion Matrix")
plt.show()
print("Naive Bayes Average Confusion Matrix:\n", conf_matrix_custom_nb)

### SVM

In [ ]:
print("\n--- Training SVM Model ---")

# Separate loop for SVM
for fold, (train_index, test_index) in enumerate(skf.split(X_text_train, y_encoded_train)):
    print(f"\nSVM - Fold {fold + 1}/{n_splits}...")

    # Ensure X_text_train_dev is in CSR format
    X_text_train_csr = X_text_train.tocsr()

    # Splits for SVM (uses text only)
    X_train_svm, X_test_svm = X_text_train_csr[train_index], X_text_train_csr[test_index]
    y_train_svm, y_test_svm = y_encoded_train[train_index], y_encoded_train[test_index]

    svm_model = SVC(kernel='linear')
    svm_model.fit(X_train_svm, y_train_svm)
    y_pred_svm = svm_model.predict(X_test_svm)
    acc_svm = accuracy_score(y_test_svm, y_pred_svm)
    results["SVM"].append(acc_svm)

    precision_svm_f, recall_svm_f, f1_svm_f, _ = precision_recall_fscore_support(y_test_svm, y_pred_svm, average='binary', pos_label=1)
    precision_list_svm.append(precision_svm_f)
    recall_list_svm.append(recall_svm_f)
    f1_list_svm.append(f1_svm_f)
    conf_matrix_sum_svm += confusion_matrix(y_test_svm, y_pred_svm)

    print(f"  SVM - Accuracy: {acc_svm:.4f}, Precision: {precision_svm_f:.4f}, Recall: {recall_svm_f:.4f}, F1 Score: {f1_svm_f:.4f}")

# Calculate and print average results for SVM
avg_accuracy_svm = np.mean(results["SVM"])
std_accuracy_svm = np.std(results["SVM"])
avg_precision_svm = np.mean(precision_list_svm)
avg_recall_svm = np.mean(recall_list_svm)
avg_f1_svm = np.mean(f1_list_svm)

print(f"\nAverage Accuracy for SVM over {n_splits} folds: {avg_accuracy_svm:.4f} ± {std_accuracy_svm:.4f}")
print(f"  Avg Precision: {avg_precision_svm:.4f}, Avg Recall: {avg_recall_svm:.4f}, Avg F1 Score: {avg_f1_svm:.4f}")

# Plotting average confusion matrix for SVM
avg_conf_matrix_svm = conf_matrix_sum_svm / n_splits
conf_matrix_custom_svm = np.array([[avg_conf_matrix_svm[1, 1], avg_conf_matrix_svm[0, 1]],
                                 [avg_conf_matrix_svm[1, 0], avg_conf_matrix_svm[0, 0]]], dtype=int)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_svm, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"],
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("SVM Average Confusion Matrix")
plt.show()
print("SVM Average Confusion Matrix:\n", conf_matrix_custom_svm)